# Build the derived HuggingFace release

Reproduces `study_gene_membership.parquet` and `studies.parquet` from the cached inputs, stages
the release folder, and shows what would be uploaded.

**This notebook does not upload.** The last cell prints the one command that does, so that
publishing is always a deliberate act rather than a side effect of running a notebook top to
bottom.

Only derived facts are redistributed: no raw supplementary file is mirrored, whatever its licence.
See [`docs/hf_release_methods.md`](../docs/hf_release_methods.md).

In [ ]:
import pandas as pd

from starplast import hf_publish, paths

# Resolved, never assumed. paths.check() reports the cache and the command that builds it if absent.
ok, message = paths.check()
print(message)
assert ok, message

## Inputs

Three cached tables. `interaction_study_members` is the membership itself; `interaction_studies`
is per-study metadata including why a study failed to parse; `study_licenses` is what could be
established about each article's licence from its own full text.

In [ ]:
members = pd.read_parquet(paths.cache_file("interaction_study_members.parquet"))
studies = pd.read_parquet(paths.cache_file("interaction_studies.parquet"))
licenses = pd.read_parquet(paths.cache_file("study_licenses.parquet"))

print(f"{len(members):,} membership rows over {members.pmid.nunique()} studies")
print(f"{len(studies)} studies catalogued, {int(studies.parsed.sum())} parsed")
print(f"{int(licenses.redistributable.sum())} of {len(licenses)} with a confirmed permissive licence")

## The membership caveat, in numbers

The reason this is published as membership rather than as an interaction network. A supplement is
usually a study's complete quantification table, not its hit list, so a row means *this gene
appears somewhere in this study's supplementary data* — a fact about a document, not a protein.

In [ ]:
per_study = members.groupby("pmid").size().sort_values(ascending=False)
print(f"median genes per study : {int(per_study.median()):,}")
print(f"studies over 2,000     : {int((per_study > 2000).sum())}")
print(f"largest single study   : {int(per_study.max()):,} genes")
print()
print("Treating a row as an interaction would manufacture tens of thousands of false edges.")

## Stage the release

Writes the two parquet files, the dataset card, the methods document and this notebook into one
folder. Nothing leaves the machine.

In [ ]:
out = hf_publish.build_release("hf_release", members, studies, licenses)

import os
for name in sorted(os.listdir(out)):
    size = os.path.getsize(os.path.join(out, name))
    print(f"  {name:<34} {size/1024:>9,.0f} KB")

## Check before publishing

That the staged folder contains only derived tables and documentation — no mirrored supplement.

In [ ]:
allowed = {".parquet", ".md", ".ipynb"}
unexpected = [f for f in os.listdir(out) if os.path.splitext(f)[1] not in allowed]
assert not unexpected, f"refusing to publish unexpected files: {unexpected}"
print("staged folder holds only derived tables and documentation")

## Upload

Deliberately not executed here. Run this from a shell when you mean to publish:

```python
from starplast import hf_publish
hf_publish.upload("einarolafsson/toxoplasma-interaction-studies", "hf_release", private=True)
```

`private=True` is the default. To make an existing dataset public, either use the Settings tab on
the dataset page or:

```python
from huggingface_hub import HfApi
HfApi().update_repo_visibility("einarolafsson/toxoplasma-interaction-studies",
                               private=False, repo_type="dataset")
```